# Temporal Graph Learning for VC Outcome Prediction
## Training Notebook (Steps 6-11)

**Run this on Google Colab with GPU runtime (T4 is fine).**

### Setup Instructions:
1. Upload `ml_investor_data.zip` (8MB) to your Google Drive root
2. Set Runtime > Change runtime type > **T4 GPU**
3. Run all cells in order

---

## 0. Setup

In [ ]:
!pip install torch-geometric xgboost sentence-transformers umap-learn -q
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu$(python -c "import torch; print(torch.version.cuda.replace('.',''))").html -q

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, GATConv, GraphConv, to_hetero
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# --- Mount Google Drive and extract data ---
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

ZIP_PATH = '/content/drive/MyDrive/ml_investor_data.zip'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'Upload ml_investor_data.zip to your Google Drive root!\n'
        f'Expected at: {ZIP_PATH}'
    )

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
    print(f'Extracted {len(z.namelist())} files')

DATA_DIR = Path('/content/data/processed')
INTERIM_DIR = Path('/content/data/interim')

print(f'Data dir: {DATA_DIR}')
print(f'Files: {sorted(os.listdir(DATA_DIR))}')

## 1. Load Preprocessed Data

In [ ]:
# Labels and splits
splits_df = pd.read_csv(INTERIM_DIR / 'splits.csv')
splits_df['trigger_funded_at'] = pd.to_datetime(splits_df['trigger_funded_at'])

# Entity indices
company_idx_df = pd.read_csv(DATA_DIR / 'company_index.csv')
investor_idx_df = pd.read_csv(DATA_DIR / 'investor_index.csv')
company_to_idx = dict(zip(company_idx_df['permalink'], company_idx_df['idx']))
investor_to_idx = dict(zip(investor_idx_df['permalink'], investor_idx_df['idx']))
n_companies = len(company_to_idx)
n_investors = len(investor_to_idx)

print(f'Labeled companies: {len(splits_df):,}')
print(f'Total companies: {n_companies:,} | Total investors: {n_investors:,}')
print(f'\nSplit distribution:')
print(splits_df['split'].value_counts())

In [ ]:
def load_split_data(split_name):
    """Load edges, company features, and investor features for a split."""
    edges = pd.read_csv(DATA_DIR / f'edges_{split_name}.csv')
    comp_feat = np.load(DATA_DIR / f'company_features_{split_name}.npy')
    inv_feat = np.load(DATA_DIR / f'investor_features_{split_name}.npy')
    return edges, comp_feat, inv_feat

def build_hetero_data(edges_df, comp_features, inv_features, splits_df, split_name):
    """Convert preprocessed data into a PyG HeteroData object."""
    data = HeteroData()
    
    # Node features
    data['company'].x = torch.tensor(comp_features, dtype=torch.float)
    data['investor'].x = torch.tensor(inv_features, dtype=torch.float)
    data['company'].num_nodes = comp_features.shape[0]
    data['investor'].num_nodes = inv_features.shape[0]
    
    # Edges (investor -> company)
    edge_index = torch.tensor(
        [edges_df['investor_idx'].values, edges_df['company_idx'].values],
        dtype=torch.long
    )
    data['investor', 'invested_in', 'company'].edge_index = edge_index
    
    # Reverse edges (company -> investor) for message passing
    data['company', 'received_from', 'investor'].edge_index = torch.stack([
        edge_index[1], edge_index[0]
    ])
    
    # Labels and masks (only for company nodes)
    subset = splits_df[splits_df['split'] == split_name]
    labels = torch.zeros(comp_features.shape[0], dtype=torch.float)
    mask = torch.zeros(comp_features.shape[0], dtype=torch.bool)
    
    for _, row in subset.iterrows():
        if row['company_permalink'] in company_to_idx:
            idx = company_to_idx[row['company_permalink']]
            labels[idx] = row['label']
            mask[idx] = True
    
    data['company'].y = labels
    data['company'].mask = mask
    
    return data

# Build all three graph snapshots
graphs = {}
for split in ['train', 'val', 'test']:
    edges, cf, ivf = load_split_data(split)
    graphs[split] = build_hetero_data(edges, cf, ivf, splits_df, split)
    g = graphs[split]
    print(f"{split}: {g['company'].mask.sum().item():,} labeled nodes | "
          f"{g['investor', 'invested_in', 'company'].edge_index.shape[1]:,} edges")

In [ ]:
# --- Graph Visualization with NetworkX ---
# Visualize a subgraph of the bipartite investor-company network.
# Full graph is too large (97K nodes), so we show the ego-network
# of a well-known company to illustrate the bipartite structure.

import networkx as nx

# Build a NetworkX bipartite graph from test edges
test_edges_df = pd.read_csv(DATA_DIR / 'edges_test.csv')
idx_to_company = {v: k for k, v in company_to_idx.items()}
idx_to_investor = {v: k for k, v in investor_to_idx.items()}

G = nx.Graph()

# Add edges (use a subset for visualization -- top investors by degree)
for _, row in test_edges_df.iterrows():
    comp = idx_to_company.get(row['company_idx'], f"C{row['company_idx']}")
    inv = idx_to_investor.get(row['investor_idx'], f"I{row['investor_idx']}")
    G.add_node(comp, bipartite=0, node_type='company')
    G.add_node(inv, bipartite=1, node_type='investor')
    G.add_edge(inv, comp)

print(f'Full graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

# Extract ego-network: pick a well-connected investor and show their portfolio
investor_degrees = {n: d for n, d in G.degree() if G.nodes[n].get('node_type') == 'investor'}
top_investors = sorted(investor_degrees, key=investor_degrees.get, reverse=True)[:5]
print(f'\nTop 5 investors by degree:')
for inv in top_investors:
    print(f'  {inv}: {investor_degrees[inv]} investments')

# Build 2-hop subgraph from the top investor
seed_investor = top_investors[0]
ego_nodes = set([seed_investor])
# 1-hop: companies invested in by seed investor
for neighbor in G.neighbors(seed_investor):
    ego_nodes.add(neighbor)
    # 2-hop: other investors in those companies (co-investors)
    for n2 in G.neighbors(neighbor):
        ego_nodes.add(n2)

# Limit to manageable size
if len(ego_nodes) > 150:
    # Keep seed investor + their companies + top co-investors
    companies = [n for n in G.neighbors(seed_investor)]
    co_investors = set()
    for c in companies:
        for inv in G.neighbors(c):
            co_investors.add(inv)
    # Keep top 15 co-investors by degree
    co_inv_sorted = sorted(co_investors, key=lambda x: G.degree(x), reverse=True)[:15]
    ego_nodes = set([seed_investor] + companies + co_inv_sorted)

subgraph = G.subgraph(ego_nodes)
print(f'\nEgo subgraph of {seed_investor}: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges')

# Visualize
fig, ax = plt.subplots(figsize=(14, 10))

node_colors = []
node_sizes = []
for n in subgraph.nodes():
    if subgraph.nodes[n].get('node_type') == 'investor':
        node_colors.append('#e74c3c')  # red for investors
        node_sizes.append(max(80, G.degree(n) * 3))
    else:
        node_colors.append('#3498db')  # blue for companies
        node_sizes.append(40)

pos = nx.spring_layout(subgraph, seed=SEED, k=1.5, iterations=50)

nx.draw_networkx_edges(subgraph, pos, alpha=0.15, width=0.5, ax=ax)
nx.draw_networkx_nodes(subgraph, pos, node_color=node_colors, node_size=node_sizes, alpha=0.7, ax=ax)

# Label only the seed investor and a few companies
labels = {}
for n in subgraph.nodes():
    if n == seed_investor:
        labels[n] = n.split('/')[-1]
    elif subgraph.nodes[n].get('node_type') == 'investor' and G.degree(n) > 50:
        labels[n] = n.split('/')[-1]
nx.draw_networkx_labels(subgraph, pos, labels, font_size=8, font_weight='bold', ax=ax)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Investor'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=10, label='Company'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=12)
ax.set_title(f'Bipartite VC Network — Ego Graph of {seed_investor.split("/")[-1]}', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig('graph_visualization.png', dpi=300, bbox_inches='tight')
plt.show()
print('Graph visualization saved.')

---
## Step 6: XGBoost Baseline

In [ ]:
def get_tabular_data(graph, split_name):
    """Extract features and labels for labeled nodes."""
    mask = graph['company'].mask
    X = graph['company'].x[mask].numpy()
    y = graph['company'].y[mask].numpy()
    return X, y

X_train, y_train = get_tabular_data(graphs['train'], 'train')
X_val, y_val = get_tabular_data(graphs['val'], 'val')
X_test, y_test = get_tabular_data(graphs['test'], 'test')

print(f'Train: {X_train.shape} | pos rate: {y_train.mean():.3f}')
print(f'Val:   {X_val.shape} | pos rate: {y_val.mean():.3f}')
print(f'Test:  {X_test.shape} | pos rate: {y_test.mean():.3f}')

In [ ]:
# Train XGBoost
scale_pos_weight = (1 - y_train.mean()) / y_train.mean()

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    max_depth=6,
    learning_rate=0.1,
    n_estimators=500,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=SEED,
    early_stopping_rounds=20,
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

# Evaluate
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_probs >= 0.5).astype(int)

xgb_results = {
    'f1': f1_score(y_test, xgb_preds),
    'roc_auc': roc_auc_score(y_test, xgb_probs),
    'precision': precision_score(y_test, xgb_preds),
    'recall': recall_score(y_test, xgb_preds),
    'accuracy': accuracy_score(y_test, xgb_preds),
}

print(f"\n{'='*50}")
print(f"  XGBoost Baseline (tabular features only)")
print(f"{'='*50}")
for k, v in xgb_results.items():
    print(f"  {k:>12}: {v:.4f}")

In [ ]:
# Feature importance with real category names
TOP_50_CATEGORIES = [
    'Software', 'Mobile', 'Biotechnology', 'E-Commerce', 'Curated Web',
    'Social Media', 'Enterprise Software', 'Advertising', 'Health Care', 'Games',
    'Internet', 'SaaS', 'Education', 'Health and Wellness', 'Apps',
    'Analytics', 'Finance', 'Technology', 'Clean Technology', 'Hardware + Software',
    'Services', 'Manufacturing', 'Medical', 'Marketplaces', 'Security',
    'Video', 'Fashion', 'Entertainment', 'Big Data', 'Consulting',
    'Real Estate', 'Cloud Computing', 'Information Technology', 'Travel', 'Social Network Media',
    'Networking', 'Search', 'Music', 'Media', 'Hospitality',
    'News', 'Retail', 'Sports', 'Startups', 'Sales and Marketing',
    'Financial Services', 'Web Hosting', 'FinTech', 'Design', 'Semiconductors',
]

feature_names = (
    ['founded_year', 'days_since_founded', 'log_total_raised', 'round_count',
     'investor_count', 'log_max_amount', 'days_since_last_round', 'is_us']
    + TOP_50_CATEGORIES
)

importance = xgb_model.feature_importances_
top_k = 15
top_idx = importance.argsort()[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_k), importance[top_idx][::-1], color='steelblue')
ax.set_yticks(range(top_k))
ax.set_yticklabels([feature_names[i] for i in top_idx][::-1])
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('XGBoost — Top 15 Features')
plt.tight_layout()
plt.show()

---
## Step 7: Node2Vec Embeddings

In [ ]:
from torch_geometric.nn import Node2Vec

def train_node2vec(graph, embedding_dim=64, walk_length=20, context_size=10,
                   walks_per_node=10, epochs=50, lr=0.01, batch_size=256):
    """Train Node2Vec on homogeneous projection of the bipartite graph."""
    homo = graph.to_homogeneous()
    
    model = Node2Vec(
        homo.edge_index,
        embedding_dim=embedding_dim,
        walk_length=walk_length,
        context_size=context_size,
        walks_per_node=walks_per_node,
        num_negative_samples=1,
        sparse=True,
    ).to(device)
    
    loader = model.loader(batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.SparseAdam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for pos_rw, neg_rw in loader:
            optimizer.zero_grad()
            loss = model.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | Loss: {total_loss/len(loader):.4f}')
    
    model.eval()
    with torch.no_grad():
        embeddings = model().cpu()
    
    # Extract company embeddings (first n_companies nodes in homogeneous graph)
    # node_type mapping: check homo.node_type
    return embeddings, homo

print('Training Node2Vec on test graph...')
n2v_embeddings, homo_graph = train_node2vec(graphs['test'])

In [ ]:
# Similarity search demo
from sklearn.metrics.pairwise import cosine_similarity

# Get company node indices from homogeneous mapping
node_types = homo_graph.node_type
company_mask_homo = (node_types == 0)  # companies typically mapped to type 0
company_embeddings = n2v_embeddings[company_mask_homo]

idx_to_company = {v: k for k, v in company_to_idx.items()}

def find_similar(company_permalink, top_k=5):
    """Find companies with most similar Node2Vec embeddings. Use exact permalink."""
    if company_permalink not in company_to_idx:
        print(f'No exact match for "{company_permalink}"')
        # Suggest close matches
        matches = [p for p in company_to_idx if company_permalink.split('/')[-1] in p]
        if matches:
            print(f'  Did you mean: {matches[:5]}')
        return
    
    idx = company_to_idx[company_permalink]
    
    if idx >= company_embeddings.shape[0]:
        print(f'Company {company_permalink} not in embedding space')
        return
    
    query = company_embeddings[idx:idx+1].numpy()
    sims = cosine_similarity(query, company_embeddings.numpy())[0]
    top_indices = sims.argsort()[-top_k-1:-1][::-1]
    
    print(f'\nCompanies similar to: {company_permalink}')
    for i, tidx in enumerate(top_indices):
        name = idx_to_company.get(tidx, '?')
        print(f'  {i+1}. {name} (similarity: {sims[tidx]:.3f})')

# Use exact permalinks
find_similar('/organization/stripe')
find_similar('/organization/airbnb')
find_similar('/organization/uber')

---
## Step 8: Sentence Embeddings (LLM Component)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load precomputed category text lookup (no raw data needed)
company_texts_df = pd.read_csv(DATA_DIR / 'company_texts.csv')
company_texts_df = company_texts_df.sort_values('idx')
texts = company_texts_df['text'].tolist()

print(f'Encoding {len(texts):,} company descriptions...')
st_model = SentenceTransformer('all-MiniLM-L6-v2')
text_embeddings = st_model.encode(texts, batch_size=256, show_progress_bar=True)
text_embeddings = torch.tensor(text_embeddings, dtype=torch.float)

print(f'Text embedding shape: {text_embeddings.shape}')
torch.save(text_embeddings, 'text_embeddings.pt')

---
## Steps 9-11: GNN Models (GCN, GraphSAGE, GAT)

### Model Definitions

In [ ]:
class GNNEncoder(torch.nn.Module):
    """Shared GNN encoder with configurable conv layer type."""
    def __init__(self, in_channels, hidden_channels=128, out_channels=64,
                 dropout=0.3, conv_type='sage'):
        super().__init__()
        self.dropout = dropout
        
        if conv_type == 'gcn':
            # GraphConv supports bipartite graphs (GCNConv does not)
            self.conv1 = GraphConv(in_channels, hidden_channels)
            self.conv2 = GraphConv(hidden_channels, out_channels)
        elif conv_type == 'sage':
            self.conv1 = SAGEConv(in_channels, hidden_channels)
            self.conv2 = SAGEConv(hidden_channels, out_channels)
        elif conv_type == 'gat':
            self.conv1 = GATConv(in_channels, hidden_channels // 4, heads=4, dropout=dropout, add_self_loops=False)
            self.conv2 = GATConv(hidden_channels, out_channels, heads=1, concat=False, dropout=dropout, add_self_loops=False)
    
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class HeteroGNN(torch.nn.Module):
    """Heterogeneous GNN for bipartite investor-company graph."""
    def __init__(self, metadata, company_in, investor_in, 
                 hidden_channels=128, out_channels=64, 
                 dropout=0.3, conv_type='sage'):
        super().__init__()
        
        # Project different node types to same dimension
        self.company_lin = torch.nn.Linear(company_in, hidden_channels)
        self.investor_lin = torch.nn.Linear(investor_in, hidden_channels)
        
        # GNN encoder (applied via to_hetero)
        self.encoder = GNNEncoder(
            in_channels=hidden_channels,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            dropout=dropout,
            conv_type=conv_type
        )
        self.encoder = to_hetero(self.encoder, metadata, aggr='sum')
        
        # Classifier head
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(out_channels, 32),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(32, 1),
        )
    
    def forward(self, x_dict, edge_index_dict):
        # Project to common dimension
        x_dict = {
            'company': F.relu(self.company_lin(x_dict['company'])),
            'investor': F.relu(self.investor_lin(x_dict['investor'])),
        }
        
        # Message passing
        x_dict = self.encoder(x_dict, edge_index_dict)
        
        # Classify company nodes
        out = self.classifier(x_dict['company']).squeeze(-1)
        return torch.sigmoid(out)

### Training Loop

In [ ]:
def train_gnn_model(model, train_graph, val_graph, epochs=200, lr=0.01, 
                    weight_decay=5e-4, patience=20):
    """Train a GNN model with early stopping on validation F1."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Class weight for imbalanced data
    train_mask = train_graph['company'].mask
    train_labels = train_graph['company'].y[train_mask]
    pos_weight = torch.tensor([(1 - train_labels.mean()) / train_labels.mean()])
    criterion = torch.nn.BCELoss(weight=None)  # sigmoid already applied in model
    
    best_val_f1 = 0
    best_epoch = 0
    best_state = None
    history = {'train_loss': [], 'val_f1': [], 'val_auc': []}
    
    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()
        
        out = model(train_graph.x_dict, train_graph.edge_index_dict)
        loss = criterion(out[train_mask], train_labels)
        loss.backward()
        optimizer.step()
        
        history['train_loss'].append(loss.item())
        
        # Validate
        model.eval()
        with torch.no_grad():
            val_out = model(val_graph.x_dict, val_graph.edge_index_dict)
            val_mask = val_graph['company'].mask
            val_labels = val_graph['company'].y[val_mask].numpy()
            val_probs = val_out[val_mask].numpy()
            val_preds = (val_probs >= 0.5).astype(int)
            
            val_f1 = f1_score(val_labels, val_preds)
            val_auc = roc_auc_score(val_labels, val_probs)
            
            history['val_f1'].append(val_f1)
            history['val_auc'].append(val_auc)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        if epoch - best_epoch >= patience:
            print(f'  Early stopping at epoch {epoch} (best: {best_epoch})')
            break
        
        if (epoch + 1) % 25 == 0:
            print(f'  Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | '
                  f'Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f}')
    
    # Restore best model
    model.load_state_dict(best_state)
    return model, history, best_epoch


def evaluate_gnn(model, graph, split_name):
    """Evaluate a trained GNN on a graph split."""
    model.eval()
    with torch.no_grad():
        out = model(graph.x_dict, graph.edge_index_dict)
        mask = graph['company'].mask
        y_true = graph['company'].y[mask].numpy()
        y_prob = out[mask].numpy()
        y_pred = (y_prob >= 0.5).astype(int)
    
    results = {
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
    }
    
    print(f"\n  {split_name} Results:")
    for k, v in results.items():
        print(f"    {k:>12}: {v:.4f}")
    
    return results, y_true, y_prob

### Train All Three GNNs

In [ ]:
metadata = graphs['train'].metadata()
company_in = graphs['train']['company'].x.shape[1]
investor_in = graphs['train']['investor'].x.shape[1]

all_results = {'XGBoost (baseline)': xgb_results}
all_histories = {}
all_predictions = {}

for conv_type, model_name in [('gcn', 'GCN'), ('sage', 'GraphSAGE'), ('gat', 'GAT')]:
    print(f"\n{'='*60}")
    print(f"  Training {model_name}")
    print(f"{'='*60}")
    
    torch.manual_seed(SEED)
    model = HeteroGNN(
        metadata=metadata,
        company_in=company_in,
        investor_in=investor_in,
        hidden_channels=128,
        out_channels=64,
        dropout=0.3,
        conv_type=conv_type,
    )
    
    model, history, best_epoch = train_gnn_model(
        model, graphs['train'], graphs['val'],
        epochs=200, lr=0.005, patience=20
    )
    
    results, y_true, y_prob = evaluate_gnn(model, graphs['test'], model_name)
    
    all_results[model_name] = results
    all_histories[model_name] = history
    all_predictions[model_name] = (y_true, y_prob)
    
    # Save model
    torch.save(model.state_dict(), f'{model_name.lower()}_model.pt')

---
## Step 12: Evaluation & Comparison

In [ ]:
# Comparison table
results_df = pd.DataFrame(all_results).T
results_df = results_df.round(4)
print('\n' + '='*70)
print('  MODEL COMPARISON (Test Set)')
print('='*70)
print(results_df.to_string())
print()

# Highlight best
for metric in ['f1', 'roc_auc']:
    best_model = results_df[metric].idxmax()
    print(f'Best {metric}: {best_model} ({results_df.loc[best_model, metric]:.4f})')

In [ ]:
# ROC Curves
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 8))

# XGBoost
fpr, tpr, _ = roc_curve(y_test, xgb_probs)
ax.plot(fpr, tpr, label=f"XGBoost (AUC={xgb_results['roc_auc']:.3f})", linewidth=2)

# GNNs
colors = {'GCN': 'orange', 'GraphSAGE': 'green', 'GAT': 'red'}
for model_name, (yt, yp) in all_predictions.items():
    fpr, tpr, _ = roc_curve(yt, yp)
    auc = all_results[model_name]['roc_auc']
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})", 
            linewidth=2, color=colors.get(model_name))

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300)
plt.show()

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, history in all_histories.items():
    axes[0].plot(history['train_loss'], label=model_name)
    axes[1].plot(history['val_f1'], label=model_name)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation F1')
axes[1].set_title('Validation F1 Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
plt.show()

---
## Save All Results

In [ ]:
# Save results locally AND to Google Drive
results_df.to_csv('model_results.csv')

for model_name, (yt, yp) in all_predictions.items():
    np.savez(f'predictions_{model_name.lower()}.npz', y_true=yt, y_prob=yp)
np.savez('predictions_xgboost.npz', y_true=y_test, y_prob=xgb_probs)

# Copy everything to Drive for persistence
import shutil
RESULTS_DIR = '/content/drive/MyDrive/ml_investor_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

for f in ['model_results.csv', 'roc_curves.png', 'training_curves.png',
          'text_embeddings.pt',
          'predictions_xgboost.npz', 'predictions_gcn.npz',
          'predictions_graphsage.npz', 'predictions_gat.npz',
          'gcn_model.pt', 'graphsage_model.pt', 'gat_model.pt']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)

print(f'All results saved to Google Drive: {RESULTS_DIR}/')
print('Files:', sorted(os.listdir(RESULTS_DIR)))

---
## V2: Improved GNN Training

**Changes from v1:**
1. Text embeddings concatenated to company features (58d -> 442d)
2. Class-weighted loss (BCEWithLogitsLoss with pos_weight)
3. Per-model learning rates (lower for GCN/GAT)
4. Learning rate scheduler (ReduceLROnPlateau)
5. Longer patience (30 epochs) and gradient clipping

In [ ]:
# --- V2 Enhancement: Concatenate text embeddings to company features ---

# Load text embeddings (generated in Step 8)
text_emb = torch.load('text_embeddings.pt')
print(f'Text embeddings: {text_emb.shape}')  # (66369, 384)

# Rebuild graphs with enriched company features
graphs_v2 = {}
for split in ['train', 'val', 'test']:
    edges, cf, ivf = load_split_data(split)
    
    # Concatenate: original 58d + text 384d = 442d
    cf_tensor = torch.tensor(cf, dtype=torch.float)
    cf_enriched = torch.cat([cf_tensor, text_emb], dim=1).numpy()
    
    graphs_v2[split] = build_hetero_data(edges, cf_enriched, ivf, splits_df, split)
    g = graphs_v2[split]
    print(f"{split}: company features {g['company'].x.shape} | "
          f"{g['investor', 'invested_in', 'company'].edge_index.shape[1]:,} edges")

In [ ]:
# --- V2 Model: same architecture, but removes sigmoid (uses BCEWithLogitsLoss) ---

class HeteroGNN_v2(torch.nn.Module):
    """Improved HeteroGNN with raw logit output for BCEWithLogitsLoss."""
    def __init__(self, metadata, company_in, investor_in, 
                 hidden_channels=128, out_channels=64, 
                 dropout=0.3, conv_type='sage'):
        super().__init__()
        
        self.company_lin = torch.nn.Linear(company_in, hidden_channels)
        self.investor_lin = torch.nn.Linear(investor_in, hidden_channels)
        
        self.encoder = GNNEncoder(
            in_channels=hidden_channels,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            dropout=dropout,
            conv_type=conv_type
        )
        self.encoder = to_hetero(self.encoder, metadata, aggr='sum')
        
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(out_channels, 32),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(32, 1),
        )
    
    def forward(self, x_dict, edge_index_dict):
        x_dict = {
            'company': F.relu(self.company_lin(x_dict['company'])),
            'investor': F.relu(self.investor_lin(x_dict['investor'])),
        }
        x_dict = self.encoder(x_dict, edge_index_dict)
        # Return raw logits (no sigmoid) -- BCEWithLogitsLoss applies it internally
        out = self.classifier(x_dict['company']).squeeze(-1)
        return out


def train_gnn_v2(model, train_graph, val_graph, epochs=300, lr=0.005, 
                 weight_decay=5e-4, patience=30):
    """V2 training: class-weighted loss, LR scheduler, gradient clipping."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-5
    )
    
    train_mask = train_graph['company'].mask
    train_labels = train_graph['company'].y[train_mask]
    
    # Class-weighted loss -- this was missing in v1!
    pos_weight = torch.tensor([(1 - train_labels.mean()) / train_labels.mean()])
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    best_val_f1 = 0
    best_epoch = 0
    best_state = None
    history = {'train_loss': [], 'val_f1': [], 'val_auc': [], 'lr': []}
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        logits = model(train_graph.x_dict, train_graph.edge_index_dict)
        loss = criterion(logits[train_mask], train_labels)
        loss.backward()
        
        # Gradient clipping -- prevents exploding gradients (the GCN spike problem)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        history['train_loss'].append(loss.item())
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(val_graph.x_dict, val_graph.edge_index_dict)
            val_mask = val_graph['company'].mask
            val_labels_np = val_graph['company'].y[val_mask].numpy()
            val_probs = torch.sigmoid(val_logits[val_mask]).numpy()
            val_preds = (val_probs >= 0.5).astype(int)
            
            val_f1 = f1_score(val_labels_np, val_preds)
            val_auc = roc_auc_score(val_labels_np, val_probs)
            
            history['val_f1'].append(val_f1)
            history['val_auc'].append(val_auc)
        
        scheduler.step(val_f1)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        if epoch - best_epoch >= patience:
            print(f'  Early stopping at epoch {epoch} (best: {best_epoch}, F1: {best_val_f1:.4f})')
            break
        
        if (epoch + 1) % 25 == 0:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | '
                  f'Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f} | LR: {cur_lr:.5f}')
    
    model.load_state_dict(best_state)
    return model, history, best_epoch


def evaluate_gnn_v2(model, graph, split_name):
    """Evaluate v2 model (logit output -> sigmoid for probabilities)."""
    model.eval()
    with torch.no_grad():
        logits = model(graph.x_dict, graph.edge_index_dict)
        mask = graph['company'].mask
        y_true = graph['company'].y[mask].numpy()
        y_prob = torch.sigmoid(logits[mask]).numpy()
        y_pred = (y_prob >= 0.5).astype(int)
    
    results = {
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
    }
    
    print(f"\n  {split_name} Results:")
    for k, v in results.items():
        print(f"    {k:>12}: {v:.4f}")
    
    return results, y_true, y_prob

print('V2 training functions ready.')

In [ ]:
# --- V2: Train all three GNNs with improvements ---

metadata_v2 = graphs_v2['train'].metadata()
company_in_v2 = graphs_v2['train']['company'].x.shape[1]  # 442 now (58 + 384)
investor_in_v2 = graphs_v2['train']['investor'].x.shape[1]  # 7

# Per-model configs: lower LR for GCN/GAT, higher hidden dim
model_configs = {
    'GCN':       {'conv_type': 'gcn',  'lr': 0.001, 'hidden': 256, 'dropout': 0.4},
    'GraphSAGE': {'conv_type': 'sage', 'lr': 0.003, 'hidden': 256, 'dropout': 0.3},
    'GAT':       {'conv_type': 'gat',  'lr': 0.001, 'hidden': 256, 'dropout': 0.3},
}

v2_results = {'XGBoost (baseline)': xgb_results}
v2_histories = {}
v2_predictions = {}

for model_name, cfg in model_configs.items():
    print(f"\n{'='*60}")
    print(f"  Training {model_name} v2 (LR={cfg['lr']}, hidden={cfg['hidden']})")
    print(f"{'='*60}")
    
    torch.manual_seed(SEED)
    model = HeteroGNN_v2(
        metadata=metadata_v2,
        company_in=company_in_v2,
        investor_in=investor_in_v2,
        hidden_channels=cfg['hidden'],
        out_channels=64,
        dropout=cfg['dropout'],
        conv_type=cfg['conv_type'],
    )
    
    model, history, best_epoch = train_gnn_v2(
        model, graphs_v2['train'], graphs_v2['val'],
        epochs=300, lr=cfg['lr'], patience=30
    )
    
    results, y_true, y_prob = evaluate_gnn_v2(model, graphs_v2['test'], model_name)
    
    v2_results[f'{model_name} v2'] = results
    v2_histories[f'{model_name} v2'] = history
    v2_predictions[f'{model_name} v2'] = (y_true, y_prob)
    
    torch.save(model.state_dict(), f'{model_name.lower()}_v2_model.pt')

# Also include v1 results for comparison
for name in ['GCN', 'GraphSAGE', 'GAT']:
    if name in all_results:
        v2_results[f'{name} v1'] = all_results[name]

In [ ]:
# --- V2 Comparison: v1 vs v2 ---

v2_df = pd.DataFrame(v2_results).T
v2_df = v2_df.round(4)

# Sort for readability
order = ['XGBoost (baseline)', 'GCN v1', 'GCN v2', 'GraphSAGE v1', 'GraphSAGE v2', 'GAT v1', 'GAT v2']
v2_df = v2_df.reindex([o for o in order if o in v2_df.index])

print('\n' + '='*75)
print('  V1 vs V2 COMPARISON (Test Set)')
print('='*75)
print(v2_df.to_string())
print()

for metric in ['f1', 'roc_auc']:
    best = v2_df[metric].idxmax()
    print(f'Best {metric}: {best} ({v2_df.loc[best, metric]:.4f})')

# ROC curves: v2 models + XGBoost
fig, ax = plt.subplots(figsize=(10, 8))

fpr, tpr, _ = roc_curve(y_test, xgb_probs)
ax.plot(fpr, tpr, label=f"XGBoost (AUC={xgb_results['roc_auc']:.3f})", linewidth=2, linestyle='--', color='blue')

colors_v2 = {'GCN v2': 'orange', 'GraphSAGE v2': 'green', 'GAT v2': 'red'}
for model_name, (yt, yp) in v2_predictions.items():
    fpr, tpr, _ = roc_curve(yt, yp)
    auc = v2_results[model_name]['roc_auc']
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})", linewidth=2, color=colors_v2.get(model_name, 'purple'))

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — V2 Models (with text embeddings + class weights)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('roc_curves_v2.png', dpi=300)
plt.show()

# Training curves v2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_name, history in v2_histories.items():
    axes[0].plot(history['train_loss'], label=model_name)
    axes[1].plot(history['val_f1'], label=model_name)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training Loss')
axes[0].set_title('V2 Training Loss'); axes[0].legend()
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation F1')
axes[1].set_title('V2 Validation F1'); axes[1].legend()
plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=300)
plt.show()

In [ ]:
# --- Save V2 results to Drive ---
v2_df.to_csv('model_results_v2.csv')

for model_name, (yt, yp) in v2_predictions.items():
    safe_name = model_name.lower().replace(' ', '_')
    np.savez(f'predictions_{safe_name}.npz', y_true=yt, y_prob=yp)

import shutil
for f in ['model_results_v2.csv', 'roc_curves_v2.png', 'training_curves_v2.png',
          'gcn_v2_model.pt', 'graphsage_v2_model.pt', 'gat_v2_model.pt',
          'predictions_gcn_v2.npz', 'predictions_graphsage_v2.npz', 'predictions_gat_v2.npz']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)

print(f'V2 results saved to {RESULTS_DIR}/')
print('Files:', sorted(os.listdir(RESULTS_DIR)))

---
## V3: Optimization Round

**Three approaches:**
1. Threshold optimization (no retraining -- find best cutoff on val set)
2. Ensemble: XGBoost + GraphSAGE v2 (average complementary signals)
3. Deeper GraphSAGE: 3 layers, residual connections, batch normalization

In [ ]:
# --- V3.1: Threshold Optimization ---
# Instead of always using 0.5, find the threshold that maximizes F1 on the validation set.
# Then apply that threshold to the test set.

from sklearn.metrics import precision_recall_curve

def find_optimal_threshold(y_true, y_prob):
    """Find threshold that maximizes F1 on given data."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    # F1 = 2 * (precision * recall) / (precision + recall)
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = f1s.argmax()
    return thresholds[best_idx], f1s[best_idx]

# Get GraphSAGE v2 validation predictions
sage_v2_model_path = 'graphsage_v2_model.pt'
torch.manual_seed(SEED)
sage_v2_model = HeteroGNN_v2(
    metadata=metadata_v2, company_in=company_in_v2, investor_in=investor_in_v2,
    hidden_channels=256, out_channels=64, dropout=0.3, conv_type='sage'
)
sage_v2_model.load_state_dict(torch.load(sage_v2_model_path))
sage_v2_model.eval()

# Val predictions for threshold search
with torch.no_grad():
    val_logits = sage_v2_model(graphs_v2['val'].x_dict, graphs_v2['val'].edge_index_dict)
    val_mask = graphs_v2['val']['company'].mask
    val_y = graphs_v2['val']['company'].y[val_mask].numpy()
    val_probs_sage = torch.sigmoid(val_logits[val_mask]).numpy()

# Test predictions
with torch.no_grad():
    test_logits = sage_v2_model(graphs_v2['test'].x_dict, graphs_v2['test'].edge_index_dict)
    test_mask = graphs_v2['test']['company'].mask
    test_y = graphs_v2['test']['company'].y[test_mask].numpy()
    test_probs_sage = torch.sigmoid(test_logits[test_mask]).numpy()

# Find optimal threshold on validation set
best_thresh, best_val_f1 = find_optimal_threshold(val_y, val_probs_sage)
print(f'Optimal threshold (from val): {best_thresh:.3f} (val F1: {best_val_f1:.4f})')
print(f'Default threshold: 0.500')

# Apply to test set
for name, thresh in [('Default (0.5)', 0.5), (f'Optimized ({best_thresh:.3f})', best_thresh)]:
    preds = (test_probs_sage >= thresh).astype(int)
    f1 = f1_score(test_y, preds)
    prec = precision_score(test_y, preds)
    rec = recall_score(test_y, preds)
    print(f'\n  {name}: F1={f1:.4f} | Precision={prec:.4f} | Recall={rec:.4f}')

In [ ]:
# --- V3.2: Ensemble (XGBoost + GraphSAGE v2) ---
# XGBoost sees tabular patterns. GraphSAGE sees graph structure.
# Their errors should be different, so averaging predictions should help.

# XGBoost test probabilities (from earlier cell)
# test_probs_sage (from cell above)

# We need to align: XGBoost predicts on labeled test nodes only,
# GraphSAGE predicts on all nodes then masks to labeled ones.
# Both test_y arrays should be identical.

# Simple average ensemble
ensemble_probs = (xgb_probs + test_probs_sage) / 2

# Weighted ensemble: search for best weight on validation set
# Get XGBoost val predictions
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]

best_w, best_ens_f1 = 0.5, 0
for w in np.arange(0.1, 0.9, 0.05):
    ens_val = w * xgb_val_probs + (1 - w) * val_probs_sage
    thresh, _ = find_optimal_threshold(y_val, ens_val)
    ens_preds = (ens_val >= thresh).astype(int)
    f1 = f1_score(y_val, ens_preds)
    if f1 > best_ens_f1:
        best_ens_f1 = f1
        best_w = w
        best_ens_thresh = thresh

print(f'Best ensemble weight: XGBoost={best_w:.2f}, GraphSAGE={1-best_w:.2f}')
print(f'Best ensemble threshold: {best_ens_thresh:.3f}')
print(f'Val F1: {best_ens_f1:.4f}')

# Apply to test
weighted_ensemble_probs = best_w * xgb_probs + (1 - best_w) * test_probs_sage

print(f'\n--- Test Results ---')
for name, probs, thresh in [
    ('XGBoost alone', xgb_probs, 0.5),
    ('GraphSAGE v2 alone (0.5)', test_probs_sage, 0.5),
    (f'GraphSAGE v2 (opt thresh)', test_probs_sage, best_thresh),
    ('Simple ensemble (50/50, 0.5)', ensemble_probs, 0.5),
    (f'Weighted ensemble ({best_w:.0%}/{1-best_w:.0%}, opt)', weighted_ensemble_probs, best_ens_thresh),
]:
    preds = (probs >= thresh).astype(int)
    results = {
        'f1': f1_score(test_y, preds),
        'roc_auc': roc_auc_score(test_y, probs),
        'precision': precision_score(test_y, preds),
        'recall': recall_score(test_y, preds),
    }
    print(f"  {name:>45}: F1={results['f1']:.4f} | AUC={results['roc_auc']:.4f} | "
          f"P={results['precision']:.4f} | R={results['recall']:.4f}")

In [ ]:
# --- V3.3: Deeper GraphSAGE with Residual Connections + BatchNorm ---

class DeepGraphSAGE(torch.nn.Module):
    """3-layer GraphSAGE with residual connections and batch normalization."""
    def __init__(self, in_channels, hidden_channels=256, out_channels=64, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, out_channels)
        
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)
        
        # Residual projection (for skip connection between conv1 and conv2)
        self.residual = torch.nn.Linear(hidden_channels, hidden_channels)
    
    def forward(self, x, edge_index):
        # Layer 1
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Layer 2 with residual connection
        identity = x
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = x + self.residual(identity)  # skip connection
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Layer 3
        x = self.conv3(x, edge_index)
        return x


class HeteroGNN_v3(torch.nn.Module):
    """Deep heterogeneous GraphSAGE with residual connections."""
    def __init__(self, metadata, company_in, investor_in, 
                 hidden_channels=256, out_channels=64, dropout=0.3):
        super().__init__()
        
        self.company_lin = torch.nn.Linear(company_in, hidden_channels)
        self.investor_lin = torch.nn.Linear(investor_in, hidden_channels)
        
        self.encoder = DeepGraphSAGE(
            in_channels=hidden_channels,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            dropout=dropout,
        )
        self.encoder = to_hetero(self.encoder, metadata, aggr='sum')
        
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(out_channels, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout * 0.5),
            torch.nn.Linear(32, 1),
        )
    
    def forward(self, x_dict, edge_index_dict):
        x_dict = {
            'company': F.relu(self.company_lin(x_dict['company'])),
            'investor': F.relu(self.investor_lin(x_dict['investor'])),
        }
        x_dict = self.encoder(x_dict, edge_index_dict)
        out = self.classifier(x_dict['company']).squeeze(-1)
        return out


# Train deep GraphSAGE
print('='*60)
print('  Training Deep GraphSAGE v3 (3 layers + residual + batchnorm)')
print('='*60)

torch.manual_seed(SEED)
v3_model = HeteroGNN_v3(
    metadata=metadata_v2,
    company_in=company_in_v2,
    investor_in=investor_in_v2,
    hidden_channels=256,
    out_channels=64,
    dropout=0.3,
)

v3_model, v3_history, v3_best_epoch = train_gnn_v2(
    v3_model, graphs_v2['train'], graphs_v2['val'],
    epochs=300, lr=0.002, patience=30
)

v3_results, v3_y_true, v3_y_prob = evaluate_gnn_v2(v3_model, graphs_v2['test'], 'Deep GraphSAGE v3')
torch.save(v3_model.state_dict(), 'graphsage_v3_model.pt')

In [ ]:
# --- V3: Final Comparison ---

# Deep GraphSAGE v3 with threshold optimization
v3_val_logits = v3_model(graphs_v2['val'].x_dict, graphs_v2['val'].edge_index_dict)
v3_val_probs = torch.sigmoid(v3_val_logits[val_mask]).detach().numpy()
v3_thresh, _ = find_optimal_threshold(val_y, v3_val_probs)

v3_opt_preds = (v3_y_prob >= v3_thresh).astype(int)

# Ensemble with v3
v3_ens_probs = best_w * xgb_probs + (1 - best_w) * v3_y_prob

# Full comparison table
print('\n' + '='*80)
print('  FINAL MODEL COMPARISON (Test Set)')
print('='*80)

final_models = {
    'XGBoost (baseline)': (xgb_probs, 0.5),
    'GraphSAGE v1': (all_predictions.get('GraphSAGE', (test_y, test_probs_sage))[1], 0.5),
    'GraphSAGE v2': (test_probs_sage, 0.5),
    f'GraphSAGE v2 (t={best_thresh:.2f})': (test_probs_sage, best_thresh),
    'Deep GraphSAGE v3': (v3_y_prob, 0.5),
    f'Deep GraphSAGE v3 (t={v3_thresh:.2f})': (v3_y_prob, v3_thresh),
    f'Ensemble XGB+SAGEv2': (weighted_ensemble_probs, best_ens_thresh),
    f'Ensemble XGB+v3': (v3_ens_probs, best_ens_thresh),
}

rows = []
for name, (probs, thresh) in final_models.items():
    preds = (probs >= thresh).astype(int)
    row = {
        'Model': name,
        'F1': f1_score(test_y, preds),
        'ROC-AUC': roc_auc_score(test_y, probs),
        'Precision': precision_score(test_y, preds),
        'Recall': recall_score(test_y, preds),
        'Accuracy': accuracy_score(test_y, preds),
    }
    rows.append(row)

final_df = pd.DataFrame(rows).set_index('Model').round(4)
print(final_df.to_string())

print(f"\nBest F1:      {final_df['F1'].idxmax()} ({final_df['F1'].max():.4f})")
print(f"Best ROC-AUC: {final_df['ROC-AUC'].idxmax()} ({final_df['ROC-AUC'].max():.4f})")

# Save final results
final_df.to_csv('model_results_final.csv')
import shutil
for f in ['model_results_final.csv', 'graphsage_v3_model.pt']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
print(f'\nFinal results saved to {RESULTS_DIR}/')

---
## Step 13: Ablation Studies

Two ablations to validate that (a) node features carry signal beyond pure topology, and (b) temporal integrity is critical.

1. **Topology-only**: Replace all node features with random vectors, keep graph structure. If GraphSAGE still performs well → graph structure alone is predictive.
2. **No temporal constraint (data leakage)**: Build graphs using ALL edges regardless of cutoff date. If metrics jump → temporal filtering is necessary to avoid inflated results.

In [ ]:
# --- Ablation 1: Topology-Only (random features, real graph) ---
# Question: Does graph structure alone carry predictive signal,
#           or do we need the handcrafted node features?
#
# Method: Replace ALL node features with random vectors (same dimension),
#         keep the exact same graph edges. Train GraphSAGE v2.
#         If F1 drops significantly → features matter.
#         If F1 stays similar → structure alone is enough.

print('='*70)
print('  ABLATION 1: Topology-Only (random features)')
print('='*70)

torch.manual_seed(SEED)
np.random.seed(SEED)

# Build graphs with random features (same shape as v2 enriched features)
graphs_topo = {}
for split in ['train', 'val', 'test']:
    edges, cf, ivf = load_split_data(split)
    
    # Random company features (same dim as enriched: 442)
    cf_enriched_dim = graphs_v2['train']['company'].x.shape[1]
    cf_random = np.random.randn(cf.shape[0], cf_enriched_dim).astype(np.float32)
    
    # Random investor features (same dim: 7)
    ivf_random = np.random.randn(ivf.shape[0], ivf.shape[1]).astype(np.float32)
    
    graphs_topo[split] = build_hetero_data(edges, cf_random, ivf_random, splits_df, split)

print(f"Company features: {graphs_topo['train']['company'].x.shape} (random)")
print(f"Investor features: {graphs_topo['train']['investor'].x.shape} (random)")
print(f"Edges (same as v2): {graphs_topo['train']['investor', 'invested_in', 'company'].edge_index.shape[1]:,}")

# Train GraphSAGE on topology-only graph
torch.manual_seed(SEED)
topo_model = HeteroGNN_v2(
    metadata=graphs_topo['train'].metadata(),
    company_in=cf_enriched_dim,
    investor_in=ivf.shape[1],
    hidden_channels=256, out_channels=64, dropout=0.3, conv_type='sage'
)

topo_model, topo_history, topo_best = train_gnn_v2(
    topo_model, graphs_topo['train'], graphs_topo['val'],
    epochs=300, lr=0.003, patience=30
)

topo_results, topo_yt, topo_yp = evaluate_gnn_v2(
    topo_model, graphs_topo['test'], 'Topology-Only GraphSAGE'
)

# Compare
print(f"\n{'─'*60}")
print(f"  Ablation 1 Summary:")
print(f"    GraphSAGE v2 (real features): F1={v2_results['GraphSAGE v2']['f1']:.4f} | AUC={v2_results['GraphSAGE v2']['roc_auc']:.4f}")
print(f"    Topology-Only (random feat):  F1={topo_results['f1']:.4f} | AUC={topo_results['roc_auc']:.4f}")
f1_drop = v2_results['GraphSAGE v2']['f1'] - topo_results['f1']
auc_drop = v2_results['GraphSAGE v2']['roc_auc'] - topo_results['roc_auc']
print(f"    Delta:                        F1={f1_drop:+.4f} | AUC={auc_drop:+.4f}")
if f1_drop > 0.05:
    print(f"  → Node features contribute significant signal beyond topology.")
else:
    print(f"  → Graph structure alone captures most of the predictive signal.")

In [ ]:
# --- Ablation 2: No Temporal Constraint (data leakage) ---
# Question: What happens if we DON'T filter edges by time?
#           i.e., use ALL edges (including future investments) for every split.
#
# This simulates the common mistake of ignoring temporal ordering.
# If metrics inflate → our temporal filtering is necessary and prevents cheating.
# This is the most important methodological validation in the paper.

print('='*70)
print('  ABLATION 2: No Temporal Constraint (data leakage)')
print('='*70)

# Use the TEST edge set (largest, includes all edges up to 2014-01-01)
# for ALL splits. This leaks future edges into train/val graphs.
edges_all, _, _ = load_split_data('test')
print(f"Using test edges for ALL splits: {len(edges_all):,} edges")
print(f"(Train normally has {graphs_v2['train']['investor', 'invested_in', 'company'].edge_index.shape[1]:,} edges)")

# Build leaked graphs: same real features (v2 enriched), but ALL edges everywhere
graphs_leaked = {}
for split in ['train', 'val', 'test']:
    _, cf, ivf = load_split_data(split)
    cf_tensor = torch.tensor(cf, dtype=torch.float)
    cf_enriched = torch.cat([cf_tensor, text_emb], dim=1).numpy()
    
    graphs_leaked[split] = build_hetero_data(edges_all, cf_enriched, ivf, splits_df, split)
    g = graphs_leaked[split]
    print(f"  {split}: {g['company'].mask.sum().item():,} labeled | "
          f"{g['investor', 'invested_in', 'company'].edge_index.shape[1]:,} edges (leaked!)")

# Train GraphSAGE on leaked graph
torch.manual_seed(SEED)
leaked_model = HeteroGNN_v2(
    metadata=graphs_leaked['train'].metadata(),
    company_in=company_in_v2,
    investor_in=investor_in_v2,
    hidden_channels=256, out_channels=64, dropout=0.3, conv_type='sage'
)

leaked_model, leaked_history, leaked_best = train_gnn_v2(
    leaked_model, graphs_leaked['train'], graphs_leaked['val'],
    epochs=300, lr=0.003, patience=30
)

leaked_results, leaked_yt, leaked_yp = evaluate_gnn_v2(
    leaked_model, graphs_leaked['test'], 'Leaked GraphSAGE'
)

# Compare
print(f"\n{'─'*60}")
print(f"  Ablation 2 Summary:")
print(f"    GraphSAGE v2 (temporal):    F1={v2_results['GraphSAGE v2']['f1']:.4f} | AUC={v2_results['GraphSAGE v2']['roc_auc']:.4f}")
print(f"    GraphSAGE (NO temporal):    F1={leaked_results['f1']:.4f} | AUC={leaked_results['roc_auc']:.4f}")
f1_diff = leaked_results['f1'] - v2_results['GraphSAGE v2']['f1']
auc_diff = leaked_results['roc_auc'] - v2_results['GraphSAGE v2']['roc_auc']
print(f"    Delta (leaked - proper):    F1={f1_diff:+.4f} | AUC={auc_diff:+.4f}")
if f1_diff > 0.02:
    print(f"  → Leaking future edges inflates metrics! Temporal filtering is essential.")
    print(f"  → Without it, the model 'sees' which companies got more investments (the answer).")
else:
    print(f"  → Surprisingly, leakage had minimal effect. Graph structure may be stable over time.")

In [ ]:
# --- Ablation Summary: Table + Bar Chart ---

ablation_rows = [
    {'Model': 'GraphSAGE v2 (full)', 'F1': v2_results['GraphSAGE v2']['f1'],
     'ROC-AUC': v2_results['GraphSAGE v2']['roc_auc'], 'Setting': 'Full model'},
    {'Model': 'Topology-Only', 'F1': topo_results['f1'],
     'ROC-AUC': topo_results['roc_auc'], 'Setting': 'Random features'},
    {'Model': 'No Temporal Filter', 'F1': leaked_results['f1'],
     'ROC-AUC': leaked_results['roc_auc'], 'Setting': 'All edges (leaked)'},
    {'Model': 'XGBoost (no graph)', 'F1': xgb_results['f1'],
     'ROC-AUC': xgb_results['roc_auc'], 'Setting': 'Tabular only'},
]

ablation_df = pd.DataFrame(ablation_rows).set_index('Model').round(4)
print('\n' + '='*70)
print('  ABLATION STUDY RESULTS')
print('='*70)
print(ablation_df.to_string())

# Bar chart (paper-ready)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models = ['Full model\n(GraphSAGE v2)', 'Topology-Only\n(random feat)', 'No Temporal\nFilter (leaked)', 'XGBoost\n(no graph)']
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']

f1_vals = [ablation_df.iloc[i]['F1'] for i in range(4)]
auc_vals = [ablation_df.iloc[i]['ROC-AUC'] for i in range(4)]

bars1 = axes[0].bar(models, f1_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('F1 Score', fontsize=12)
axes[0].set_title('Ablation: F1 Score', fontsize=13)
axes[0].set_ylim(0, max(f1_vals) * 1.15)
for bar, val in zip(bars1, f1_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

bars2 = axes[1].bar(models, auc_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('ROC-AUC', fontsize=12)
axes[1].set_title('Ablation: ROC-AUC', fontsize=13)
axes[1].set_ylim(0, max(auc_vals) * 1.15)
for bar, val in zip(bars2, auc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('ablation_chart.png', dpi=300, bbox_inches='tight')
plt.show()

# Save ablation results
ablation_df.to_csv('ablation_results.csv')
import shutil
for f in ['ablation_results.csv', 'ablation_chart.png']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
print(f'\nAblation results saved to {RESULTS_DIR}/')

---
## Step 14: Error Analysis

Breakdown of model performance by subgroups: geography, sector, funding amount, and graph connectivity (cold-start).

In [ ]:
# --- Error Analysis: Build test set analysis dataframe ---
# We need company-level features aligned with predictions for subgroup analysis.
# Using the best model: Ensemble XGB+SAGEv2

# Get test company indices and features
test_subset = splits_df[splits_df['split'] == 'test'].copy()
test_subset = test_subset[test_subset['company_permalink'].isin(company_to_idx)]
test_subset['node_idx'] = test_subset['company_permalink'].map(company_to_idx)
test_subset = test_subset.sort_values('node_idx').reset_index(drop=True)

# Load original (non-enriched) test features for interpretable columns
_, cf_test_orig, _ = load_split_data('test')

# Map features back to test companies
test_feat = cf_test_orig[test_subset['node_idx'].values]  # (n_test, 58)

# Attach key features
test_subset['is_us'] = test_feat[:, 7]  # feature index 7 = is_us
test_subset['log_total_raised'] = test_feat[:, 2]
test_subset['investor_count'] = test_feat[:, 4]
test_subset['round_count'] = test_feat[:, 3]

# Attach predictions (ensemble)
test_subset['y_true'] = y_test
test_subset['xgb_prob'] = xgb_probs
test_subset['sage_prob'] = test_probs_sage
test_subset['ens_prob'] = weighted_ensemble_probs
test_subset['ens_pred'] = (weighted_ensemble_probs >= best_ens_thresh).astype(int)
test_subset['xgb_pred'] = xgb_preds
test_subset['sage_pred'] = (test_probs_sage >= 0.5).astype(int)

# Compute correctness
test_subset['ens_correct'] = (test_subset['ens_pred'] == test_subset['y_true']).astype(int)
test_subset['xgb_correct'] = (test_subset['xgb_pred'] == test_subset['y_true']).astype(int)

# Count edges per company in test graph (graph connectivity)
test_edges = pd.read_csv(DATA_DIR / 'edges_test.csv')
edge_counts = test_edges.groupby('company_idx').size().reset_index(name='edge_count')
test_subset = test_subset.merge(edge_counts, left_on='node_idx', right_on='company_idx', how='left')
test_subset['edge_count'] = test_subset['edge_count'].fillna(0).astype(int)

# Category columns (top 50)
for i, cat in enumerate(TOP_50_CATEGORIES):
    test_subset[f'cat_{cat}'] = test_feat[:, 8 + i]

print(f'Error analysis dataframe: {test_subset.shape}')
print(f"Columns: {list(test_subset.columns[:15])}...")
print(f"\nEnsemble accuracy: {test_subset['ens_correct'].mean():.3f}")
print(f"XGBoost accuracy: {test_subset['xgb_correct'].mean():.3f}")

In [ ]:
# --- Error Analysis: Subgroup Breakdowns ---

def subgroup_metrics(df, group_col, model_pred='ens_pred', model_prob='ens_prob'):
    """Compute F1 and AUC per subgroup."""
    rows = []
    for name, grp in df.groupby(group_col):
        if len(grp) < 20 or grp['y_true'].nunique() < 2:
            continue
        rows.append({
            'Subgroup': name,
            'N': len(grp),
            'Pos Rate': grp['y_true'].mean(),
            'F1': f1_score(grp['y_true'], grp[model_pred]),
            'AUC': roc_auc_score(grp['y_true'], grp[model_prob]),
        })
    return pd.DataFrame(rows).round(4)

# --- 1. Geography: US vs non-US ---
test_subset['geography'] = test_subset['is_us'].apply(lambda x: 'US' if x > 0 else 'Non-US')
geo_df = subgroup_metrics(test_subset, 'geography')
print('='*60)
print('  GEOGRAPHY BREAKDOWN (Ensemble)')
print('='*60)
print(geo_df.to_string(index=False))

# --- 2. Funding Amount ---
test_subset['funding_bucket'] = pd.cut(
    test_subset['log_total_raised'],
    bins=[-999, 0, 0.5, 1.0, 999],
    labels=['Zero/Unknown', 'Low (<$3K)', 'Medium ($3K-$10K)', 'High (>$10K)']
)
funding_df = subgroup_metrics(test_subset, 'funding_bucket')
print(f"\n{'='*60}")
print('  FUNDING AMOUNT BREAKDOWN (Ensemble)')
print('='*60)
print(funding_df.to_string(index=False))

# --- 3. Cold-Start: Graph Connectivity ---
test_subset['connectivity'] = pd.cut(
    test_subset['edge_count'],
    bins=[-1, 0, 1, 3, 999],
    labels=['Isolated (0 edges)', '1 edge', '2-3 edges', '4+ edges']
)
cold_df = subgroup_metrics(test_subset, 'connectivity')
print(f"\n{'='*60}")
print('  COLD-START ANALYSIS (Ensemble)')
print('='*60)
print(cold_df.to_string(index=False))

# Compare XGBoost vs Ensemble on cold-start companies
cold_start = test_subset[test_subset['edge_count'] == 0]
connected = test_subset[test_subset['edge_count'] > 0]

if len(cold_start) > 20 and cold_start['y_true'].nunique() == 2:
    print(f"\n  Cold-start (0 edges): N={len(cold_start)}")
    print(f"    XGBoost F1:  {f1_score(cold_start['y_true'], cold_start['xgb_pred']):.4f}")
    print(f"    Ensemble F1: {f1_score(cold_start['y_true'], cold_start['ens_pred']):.4f}")
    print(f"  Connected (1+ edges): N={len(connected)}")
    print(f"    XGBoost F1:  {f1_score(connected['y_true'], connected['xgb_pred']):.4f}")
    print(f"    Ensemble F1: {f1_score(connected['y_true'], connected['ens_pred']):.4f}")

# --- 4. Top Sectors ---
sector_rows = []
for cat in TOP_50_CATEGORIES[:20]:
    col = f'cat_{cat}'
    mask = test_subset[col] > 0
    grp = test_subset[mask]
    if len(grp) >= 30 and grp['y_true'].nunique() == 2:
        sector_rows.append({
            'Sector': cat,
            'N': len(grp),
            'Pos Rate': grp['y_true'].mean(),
            'Ens F1': f1_score(grp['y_true'], grp['ens_pred']),
            'XGB F1': f1_score(grp['y_true'], grp['xgb_pred']),
            'Gain': f1_score(grp['y_true'], grp['ens_pred']) - f1_score(grp['y_true'], grp['xgb_pred']),
        })

sector_df = pd.DataFrame(sector_rows).round(4).sort_values('Gain', ascending=False)
print(f"\n{'='*60}")
print('  SECTOR BREAKDOWN: Ensemble vs XGBoost (top 20 categories)')
print('='*60)
print(sector_df.to_string(index=False))

---
## Step 15: Visualizations

UMAP projection of GraphSAGE learned embeddings and error analysis plots.

In [ ]:
# --- UMAP of GraphSAGE Learned Embeddings ---
import umap

# Extract company embeddings from the trained GraphSAGE v2 model
# We hook into the model just before the classifier
sage_v2_model.eval()
with torch.no_grad():
    x_dict = {
        'company': F.relu(sage_v2_model.company_lin(graphs_v2['test']['company'].x)),
        'investor': F.relu(sage_v2_model.investor_lin(graphs_v2['test']['investor'].x)),
    }
    x_dict = sage_v2_model.encoder(x_dict, graphs_v2['test'].edge_index_dict)
    company_repr = x_dict['company'].numpy()  # (66369, 64)

# Get labeled test companies only
test_mask_np = graphs_v2['test']['company'].mask.numpy()
test_repr = company_repr[test_mask_np]  # (5989, 64)
test_labels = graphs_v2['test']['company'].y[test_mask_np].numpy()

print(f'Embedding shape: {test_repr.shape}')
print(f'Running UMAP...')

reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=30, min_dist=0.3)
embedding_2d = reducer.fit_transform(test_repr)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# By true label
scatter1 = axes[0].scatter(
    embedding_2d[:, 0], embedding_2d[:, 1],
    c=test_labels, cmap='RdYlGn', alpha=0.4, s=8, edgecolors='none'
)
axes[0].set_title('UMAP of GraphSAGE Embeddings\n(colored by true label)', fontsize=13)
axes[0].set_xlabel('UMAP-1')
axes[0].set_ylabel('UMAP-2')
cbar1 = plt.colorbar(scatter1, ax=axes[0], shrink=0.8)
cbar1.set_ticks([0, 1])
cbar1.set_ticklabels(['No follow-on', 'Raised follow-on'])

# By prediction confidence (ensemble)
scatter2 = axes[1].scatter(
    embedding_2d[:, 0], embedding_2d[:, 1],
    c=weighted_ensemble_probs, cmap='RdYlGn', alpha=0.4, s=8, edgecolors='none'
)
axes[1].set_title('UMAP of GraphSAGE Embeddings\n(colored by ensemble probability)', fontsize=13)
axes[1].set_xlabel('UMAP-1')
axes[1].set_ylabel('UMAP-2')
cbar2 = plt.colorbar(scatter2, ax=axes[1], shrink=0.8)
cbar2.set_label('P(follow-on)')

plt.tight_layout()
plt.savefig('umap_embeddings.png', dpi=300, bbox_inches='tight')
plt.show()
print('UMAP visualization saved.')

In [ ]:
# --- Error Analysis Visualization ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Cold-start bar chart: XGBoost vs Ensemble by connectivity
connectivity_groups = ['Isolated (0 edges)', '1 edge', '2-3 edges', '4+ edges']
xgb_f1s = []
ens_f1s = []
for grp_name in connectivity_groups:
    grp = test_subset[test_subset['connectivity'] == grp_name]
    if len(grp) >= 20 and grp['y_true'].nunique() == 2:
        xgb_f1s.append(f1_score(grp['y_true'], grp['xgb_pred']))
        ens_f1s.append(f1_score(grp['y_true'], grp['ens_pred']))
    else:
        xgb_f1s.append(0)
        ens_f1s.append(0)

x = np.arange(len(connectivity_groups))
w = 0.35
axes[0].bar(x - w/2, xgb_f1s, w, label='XGBoost', color='#3498db')
axes[0].bar(x + w/2, ens_f1s, w, label='Ensemble', color='#2ecc71')
axes[0].set_xticks(x)
axes[0].set_xticklabels(connectivity_groups, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Cold-Start Analysis:\nF1 by Graph Connectivity')
axes[0].legend()

# 2. Geography
geo_groups = ['US', 'Non-US']
xgb_geo = []
ens_geo = []
for g in geo_groups:
    grp = test_subset[test_subset['geography'] == g]
    if len(grp) >= 20 and grp['y_true'].nunique() == 2:
        xgb_geo.append(f1_score(grp['y_true'], grp['xgb_pred']))
        ens_geo.append(f1_score(grp['y_true'], grp['ens_pred']))
    else:
        xgb_geo.append(0)
        ens_geo.append(0)

x2 = np.arange(len(geo_groups))
axes[1].bar(x2 - w/2, xgb_geo, w, label='XGBoost', color='#3498db')
axes[1].bar(x2 + w/2, ens_geo, w, label='Ensemble', color='#2ecc71')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(geo_groups)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Geography:\nF1 by Region')
axes[1].legend()

# 3. Prediction distribution (histogram of probabilities)
axes[2].hist(weighted_ensemble_probs[y_test == 1], bins=30, alpha=0.6, label='Positive (raised)', color='green', density=True)
axes[2].hist(weighted_ensemble_probs[y_test == 0], bins=30, alpha=0.6, label='Negative (no raise)', color='red', density=True)
axes[2].axvline(x=best_ens_thresh, color='black', linestyle='--', label=f'Threshold={best_ens_thresh:.2f}')
axes[2].set_xlabel('Ensemble Probability')
axes[2].set_ylabel('Density')
axes[2].set_title('Prediction Distribution\n(Ensemble)')
axes[2].legend()

plt.tight_layout()
plt.savefig('error_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Save all new figures to Drive
import shutil
for f in ['umap_embeddings.png', 'error_analysis.png', 'ablation_chart.png']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
print(f'All figures saved to {RESULTS_DIR}/')

---
## Step 16: Link Prediction with LLMs (Sentence-Transformers)

Use an open-source transformer model (**all-MiniLM-L6-v2**) to predict whether an investor–company link exists in the bipartite graph.

**Approach:**
1. Represent each company by its sector/category embedding (sentence-transformer)
2. Represent each investor by the mean embedding of their portfolio companies
3. Create positive samples (real edges) and negative samples (random non-edges)
4. Predict link existence via cosine similarity + logistic regression
5. Evaluate on held-out edges with F1, AUC, Precision, Recall

In [ ]:
# --- Link Prediction with Sentence-Transformer Embeddings ---
# Uses the sentence-transformer model (already loaded in Step 8) to build
# investor and company representations, then predicts link existence.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# --- 1. Build company text representations ---
# company_texts.csv has (idx, permalink, text) where text = category_list

# Reuse the sentence-transformer model from Step 8
try:
    st_model
    print("Reusing sentence-transformer model from Step 8.")
except NameError:
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Loaded sentence-transformer: all-MiniLM-L6-v2")

company_texts_df = pd.read_csv(DATA_DIR / 'company_texts.csv')
company_texts_df = company_texts_df.sort_values('idx').reset_index(drop=True)
company_texts_df['text'] = company_texts_df['text'].fillna('Unknown')

print(f"Encoding {len(company_texts_df)} company categories with sentence-transformer...")
company_embeddings_lp = st_model.encode(
    company_texts_df['text'].tolist(),
    show_progress_bar=True, batch_size=256
)
print(f"Company embeddings shape: {company_embeddings_lp.shape}")

# --- 2. Build investor representations ---
# Each investor = mean embedding of their portfolio companies (train edges only)

train_edges_lp = pd.read_csv(DATA_DIR / 'edges_train.csv')
investor_index_df = pd.read_csv(DATA_DIR / 'investor_index.csv')
n_investors = len(investor_index_df)

print(f"Building investor profiles from {len(train_edges_lp)} train edges...")

investor_portfolios = {}
for _, row in train_edges_lp.iterrows():
    inv_idx = int(row['investor_idx'])
    comp_idx = int(row['company_idx'])
    if comp_idx < len(company_embeddings_lp):
        if inv_idx not in investor_portfolios:
            investor_portfolios[inv_idx] = []
        investor_portfolios[inv_idx].append(comp_idx)

# Compute mean embedding per investor
investor_embeddings_lp = np.zeros((n_investors, company_embeddings_lp.shape[1]))
for inv_idx, portfolio in investor_portfolios.items():
    investor_embeddings_lp[inv_idx] = company_embeddings_lp[portfolio].mean(axis=0)

n_with_portfolio = len(investor_portfolios)
print(f"Investors with portfolios: {n_with_portfolio} / {n_investors}")
print(f"Investor embeddings shape: {investor_embeddings_lp.shape}")

In [ ]:
# --- 3. Create positive and negative edge samples ---
# Positive: real edges from TEST set (edges the model hasn't seen during embedding)
# Negative: random investor-company pairs that DON'T exist in any split

test_edges_lp = pd.read_csv(DATA_DIR / 'edges_test.csv')

# Build set of ALL known edges across splits
all_edges_set = set()
for split in ['train', 'val', 'test']:
    e = pd.read_csv(DATA_DIR / f'edges_{split}.csv')
    for _, row in e.iterrows():
        all_edges_set.add((int(row['investor_idx']), int(row['company_idx'])))

# Filter test edges: only keep pairs where investor has a portfolio embedding
pos_pairs = []
for _, row in test_edges_lp.iterrows():
    inv_idx = int(row['investor_idx'])
    comp_idx = int(row['company_idx'])
    if inv_idx in investor_portfolios and comp_idx < len(company_embeddings_lp):
        pos_pairs.append((inv_idx, comp_idx))

print(f"Positive test edges (with embeddings): {len(pos_pairs)}")

# Sample equal number of negative pairs
np.random.seed(SEED)
neg_pairs = []
inv_with_emb = list(investor_portfolios.keys())
n_companies = len(company_embeddings_lp)
max_attempts = len(pos_pairs) * 20
attempts = 0
while len(neg_pairs) < len(pos_pairs) and attempts < max_attempts:
    inv_idx = int(np.random.choice(inv_with_emb))
    comp_idx = int(np.random.randint(0, n_companies))
    if (inv_idx, comp_idx) not in all_edges_set:
        neg_pairs.append((inv_idx, comp_idx))
        all_edges_set.add((inv_idx, comp_idx))
    attempts += 1

print(f"Negative pairs sampled: {len(neg_pairs)}")

# --- 4. Build feature matrix ---
from numpy.linalg import norm

def pair_features_lp(inv_idx, comp_idx):
    inv_emb = investor_embeddings_lp[inv_idx]
    comp_emb = company_embeddings_lp[comp_idx]
    cos_sim = np.dot(inv_emb, comp_emb) / (norm(inv_emb) * norm(comp_emb) + 1e-8)
    return np.concatenate([inv_emb, comp_emb, [cos_sim]])

X_link = np.array([pair_features_lp(i, c) for i, c in pos_pairs + neg_pairs])
y_link = np.array([1]*len(pos_pairs) + [0]*len(neg_pairs))

# Shuffle
shuffle_idx = np.random.permutation(len(X_link))
X_link = X_link[shuffle_idx]
y_link = y_link[shuffle_idx]

# Train/test split (80/20)
split_pt = int(0.8 * len(X_link))
X_lp_train, X_lp_test = X_link[:split_pt], X_link[split_pt:]
y_lp_train, y_lp_test = y_link[:split_pt], y_link[split_pt:]

print(f"\nLink prediction dataset: {len(X_link)} pairs ({len(pos_pairs)} pos, {len(neg_pairs)} neg)")
print(f"Train: {len(X_lp_train)}, Test: {len(X_lp_test)}")

# --- 5. Train logistic regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_model.fit(X_lp_train, y_lp_train)

y_lp_pred = lr_model.predict(X_lp_test)
y_lp_prob = lr_model.predict_proba(X_lp_test)[:, 1]

# --- 6. Evaluate ---
lp_f1 = f1_score(y_lp_test, y_lp_pred)
lp_auc = roc_auc_score(y_lp_test, y_lp_prob)
lp_precision = precision_score(y_lp_test, y_lp_pred)
lp_recall = recall_score(y_lp_test, y_lp_pred)

print("\n" + "="*50)
print("LINK PREDICTION RESULTS (Sentence-Transformer + LR)")
print("="*50)
print(f"F1 Score:    {lp_f1:.4f}")
print(f"ROC-AUC:     {lp_auc:.4f}")
print(f"Precision:   {lp_precision:.4f}")
print(f"Recall:      {lp_recall:.4f}")
print("\nClassification Report:")
print(classification_report(y_lp_test, y_lp_pred, target_names=['No Link', 'Link']))

# --- 7. Cosine similarity baseline ---
cos_sims_test = X_lp_test[:, -1]
cos_thresh = np.median(cos_sims_test)
cos_pred = (cos_sims_test >= cos_thresh).astype(int)

cos_f1 = f1_score(y_lp_test, cos_pred)
cos_auc = roc_auc_score(y_lp_test, cos_sims_test)

print(f"\nBaseline (cosine similarity only):")
print(f"  F1: {cos_f1:.4f}  |  AUC: {cos_auc:.4f}")

# Summary table
lp_results = pd.DataFrame([
    {'Method': 'Cosine Similarity Only', 'F1': cos_f1, 'AUC': cos_auc},
    {'Method': 'LLM Embeddings + LR', 'F1': lp_f1, 'AUC': lp_auc},
])
print("\n")
print(lp_results.to_string(index=False))

lp_results.to_csv(RESULTS_DIR / 'link_prediction_results.csv', index=False)
print(f"\nResults saved to {RESULTS_DIR}/link_prediction_results.csv")

In [ ]:
# --- Link Prediction Visualization ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Cosine similarity distributions for positive vs negative pairs
cos_pos = X_lp_test[y_lp_test == 1, -1]
cos_neg = X_lp_test[y_lp_test == 0, -1]
axes[0].hist(cos_pos, bins=30, alpha=0.6, label='Real Links', color='green', density=True)
axes[0].hist(cos_neg, bins=30, alpha=0.6, label='Non-Links', color='red', density=True)
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Density')
axes[0].set_title('Embedding Similarity Distribution\n(Investor vs Company)')
axes[0].legend()

# 2. ROC curve for link prediction
from sklearn.metrics import roc_curve
fpr_lp, tpr_lp, _ = roc_curve(y_lp_test, y_lp_prob)
fpr_cos, tpr_cos, _ = roc_curve(y_lp_test, cos_sims_test)
axes[1].plot(fpr_lp, tpr_lp, label=f'LLM + LR (AUC={lp_auc:.3f})', linewidth=2)
axes[1].plot(fpr_cos, tpr_cos, label=f'Cosine Only (AUC={cos_auc:.3f})', linewidth=2, linestyle='--')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Link Prediction ROC Curve')
axes[1].legend()

# 3. Comparison bar chart
methods = ['Cosine\nSimilarity', 'LLM\nEmbeddings + LR']
f1_vals = [cos_f1, lp_f1]
auc_vals = [cos_auc, lp_auc]
x = np.arange(len(methods))
w = 0.3
axes[2].bar(x - w/2, f1_vals, w, label='F1', color='#3498db')
axes[2].bar(x + w/2, auc_vals, w, label='AUC', color='#2ecc71')
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods)
axes[2].set_ylabel('Score')
axes[2].set_title('Link Prediction: Method Comparison')
axes[2].legend()
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('link_prediction.png', dpi=300, bbox_inches='tight')
plt.show()

import shutil
shutil.copy('link_prediction.png', RESULTS_DIR)
print(f"Link prediction figure saved to {RESULTS_DIR}/link_prediction.png")